### 8. Qual é a taxa de concessão de tutelas de urgência cautelares antecedentes ao pedido principal de recuperação?

**Pergunta:** antes (ou no limiar) do pedido de recuperação, o devedor pede
ao juiz uma tutela de urgência — tipicamente a suspensão das execuções —
para ganhar tempo e viabilizar a negociação. Quantas RJs têm essa tutela?

**Fonte no mart:** `gold_base_analitica` —
`data_primeira_tutela` (primeira tutela de urgência/cautelar registrada nos
autos) e `data_processamento_efetivo` (concessão da RJ, pergunta 1); a
antecedência é computada comparando as duas datas. A via autônoma é testada
pelas classes 12134 e 12084 (ambas denominadas Tutela Cautelar Antecedente), vinculada à RJ pela chave do número CNJ (raiz + origem, método validado na pergunta 7). Este
notebook não refaz cálculos: consome o resultado pronto.

**Recorte:** classe 129, ajuizado a partir de 09/06/2005. A tutela é
instrumento do CPC, não da Lei 11.101 — não há filtro adicional de lei.

**Sobre "concessão":** o dado registra a tutela
**materializada nos autos** (com data). Pedidos denegados tendem a não
deixar marco datado — então a taxa medida é de **deferimento**, não de
pedido.

**Limitações:**
1. A coluna data_primeira_tutela cobre apenas tutelas registradas nos próprios autos da RJ — tutelas deferidas em autos apensados ou em incidentes podem não aparecer nela;
2. Tutelas ajuizadas em unidade de origem diversa da RJ não casam na chave CNJ (subcontagem);
3. **Registros incompletos**: entre as RJs com tutela registrada, 266 (34%) não têm data de processamento registrada — o evento secundário (a tutela de urgência) aparece e o principal não. Isso impede classificar a antecedência da tutela nesses casos. É o mesmo padrão de registros incompletos já documentado nas perguntas 1 e 2.

**Observação — duas vias distintas:** este notebook examina duas vias para a mesma pergunta. Na **via intra-autos**, a tutela tramita dentro dos próprios autos da RJ (medida pela coluna
data_primeira_tutela). Na **via autônoma**, a tutela tem processo
próprio e separado (classes 12134/12084) — e, como o dado não declara a qual RJ ela pertence, o vínculo é deduzido pela estrutura do número CNJ (raiz + origem). Os números das duas vias não se somam: são universos diferentes.

#### O que a consulta a seguir mede e como (via intra-autos)

**Entrada:** todos os campos vêm da tabela
`default.gold_base_analitica`, já filtrada para os processos de Recuperação
Judicial (classe 129) ajuizados no período da lei vigente (a partir de
09/06/2005). 

A consulta lê três colunas por processo:

| Coluna lida | O que representa |
|---|---|
| `data_ajuizamento` | quando a RJ foi protocolada (o "pedido principal") |
| `data_primeira_tutela` | a data da primeira tutela de urgência registrada nos próprios autos (nulo = o processo não tem tutela) |
| `data_processamento_efetivo` | a data em que o juiz concedeu o processamento da RJ (nulo = não registrada) |

**Saída — o que ela produz (uma única linha com seis números):**

| Coluna produzida | Como é calculada | O que significa |
|---|---|---|
| `total_rj` | conta todos os processos do recorte | o denominador de todas as taxas (5.709) |
| `com_tutela` | conta os processos em que a data da tutela existe | quantas RJs têm tutela de urgência nos próprios autos (792) |
| `pct_rj_com_tutela` | com_tutela ÷ total_rj × 100 | a taxa de incidência da tutela (13,9%) |
| `tutela_no_ajuizamento` | conta os casos em que a tutela é igual ao ajuizamento | quantas tutelas vieram no mesmo dia do pedido principal — (12) |
| `mediana_dias_ajuizamento_tutela` | mediana de (data da tutela − data de ajuizamento), só para quem tem tutela | quantos dias após o pedido a tutela típica é deferida (+26 dias) |
| `com_tutela_sem_processamento_datado` | conta os casos com tutela mas **sem** data de processamento | quantas RJs têm o evento secundário (tutela) registrado e o principal (processamento) não (266) — alimenta a limitação 3 sobre registros incompletos |

**Em uma frase:** a consulta pega os 5.709 processos de RJ do período da lei vigente, verifica em quantos há tutela de urgência nos próprios autos e — para esses — situa a tutela em relação à data do pedido e verifica em quantos o processamento está datado, produzindo os seis números que sustentam a resposta: a incidência (13,9%), as tutelas deferidas no próprio dia do pedido (12), o tempo típico até a tutela (26 dias) e o balanço de registros incompletos (266).

In [0]:
%sql
-- Q8 · VIA INTRA-AUTOS: incidencia da tutela e antecedencia ao pedido principal
SELECT
  count(*)                                                              AS total_rj,
  count(CASE WHEN data_primeira_tutela IS NOT NULL THEN 1 END)          AS com_tutela,
  round(count(CASE WHEN data_primeira_tutela IS NOT NULL THEN 1 END)
        * 100.0 / count(*), 1)                                          AS pct_rj_com_tutela,
  count(CASE WHEN data_primeira_tutela = data_ajuizamento THEN 1 END)   AS tutela_no_ajuizamento,
  round(percentile_approx(
          CASE WHEN data_primeira_tutela IS NOT NULL
                THEN datediff(data_primeira_tutela, data_ajuizamento) END, 0.5), 1)
                                                                        AS mediana_dias_ajuizamento_tutela,
  count(CASE WHEN data_primeira_tutela IS NOT NULL
              AND data_processamento_efetivo IS NULL THEN 1 END)        AS com_tutela_sem_processamento_datado
FROM default.gold_base_analitica
WHERE classe_codigo = 129 AND escopo_lei_atual;

#### O que a consulta a seguir mede e como (via autônoma)

**Entrada:** lê a mesma tabela
`default.gold_base_analitica`, mas agora em **duas frentes**:

| Frente | Filtro | O que extrai |
|---|---|---|
| CTE `rj` | classe 129 (pedidos de RJ) no período da lei vigente | de cada RJ: a **raiz** (7 primeiros dígitos do número CNJ), a **origem** (4 últimos dígitos, que identificam a vara) e a data de ajuizamento |
| CTE `tutelas` | classes 12134 (tutela cautelar antecedente) e 12084 (tutela cautelar antecedente) | de cada tutela ajuizada em **autos próprios**: o número do processo, a mesma raiz, a mesma origem e a data de ajuizamento |

A liga entre as duas frentes é a **estrutura do número CNJ**: processos
relacionados (um incidente e o seu processo-mãe) compartilham a raiz e a
origem — só o número sequencial do meio muda. Assim, uma tutela autônoma
é considerada **vinculada** a uma RJ quando existe, no corpus, uma
recuperação com a mesma raiz + origem.

**Saída — o que ela produz (uma única linha com três números):**

| Coluna produzida | Como é calculada | O que significa |
|---|---|---|
| `total_tutelas_autonomas` | conta todas as tutelas das classes 12134/12084 no corpus | o universo da via autônoma (279) |
| `vinculadas_a_rj` | conta as tutelas que encontraram uma RJ com a mesma raiz + origem no JOIN | quantas dessas tutelas pertencem a uma RJ do corpus (1) |
| `antecedentes` | conta as vinculadas em que a tutela é anterior **ou igual** ao ajuizamento da RJ | quantas foram usadas como **preparação** da recuperação — a resposta da via autônoma (0; o único caso é posterior) |


**Em uma frase:** a consulta pega as 279 tutelas ajuizadas em processos separados (a via autônoma do art. 303 do CPC), verifica quais delas
pertencem a uma RJ do corpus pela chave do número CNJ (mesma raiz e
mesma origem) e produz três números: o universo autônomo (279), os
vínculos encontrados (1) e os casos de verdadeira preparação, anteriores ao pedido (0) — confirmando o uso de tutela cautelar antecedente, uma ação autônoma, não é uma via normalmente utilizada em recuperação judicial.

In [0]:
%sql
-- Q8 · VIA AUTONOMA: tutelas cautelares antecedentes (12134/12084) vinculadas a RJ
WITH rj AS (
  SELECT substring(numero_processo, 1, 7)  AS raiz,
         substring(numero_processo, 17, 4) AS origem,
         data_ajuizamento                  AS data_ajuizamento_rj
  FROM default.gold_base_analitica
  WHERE classe_codigo = 129 AND escopo_lei_atual
),
tutelas AS (
  SELECT numero_processo,
         substring(numero_processo, 1, 7)  AS raiz,
         substring(numero_processo, 17, 4) AS origem,
         data_ajuizamento
  FROM default.gold_base_analitica
  WHERE classe_codigo IN (12134, 12084)
)
SELECT
  (SELECT count(*) FROM tutelas)                                          AS total_tutelas_autonomas,
  count(t.numero_processo)                                                AS vinculadas_a_rj,
  count(CASE WHEN t.data_ajuizamento <= r.data_ajuizamento_rj
             THEN 1 END)                                                  AS antecedentes
FROM tutelas t
JOIN rj r ON r.raiz = t.raiz AND r.origem = t.origem;

#### Análise da resposta (pergunta 8)

**Resultados (classe 129-recuperação judicial, Lei 11.101/2005, 5.709 RJs):**
| Indicador | Valor |
|---|---|
| Total de RJs | 5709 |
| RJs com tutela registrada nos próprios autos | 792 (**13,9%**) |
| RJ com tutela no dia do ajuizamento do pedido principal | **12 (0,2%)** |
| Mediana ajuizamento → tutela | **+26 dias** |
| Tutelas autônomas no corpus (classes 12134/12084) | 279 |
| Autônomas vinculadas a uma RJ | 1 |

**Três achados:**

1. **A tutela pedida antes da recuperação é praticamente inexistente:**
   de todas as 5.709 RJs, apenas 12 (0,2%) tiveram uma tutela deferida
   **no próprio dia do ajuizamento do pedido** — o máximo que a via
   intra-autos permite como "antecedência".

2. **O que existe na prática é a tutela pedida junto com a recuperação:**
   13,9% das RJs registram uma tutela de urgência dentro dos próprios
   autos, deferida em geral 26 dias após o pedido. O caminho típico é:
   o empresário entra com a recuperação, o juiz analisa o pedido e, na
   mesma decisão (ou nos dias seguintes a ela), concede a proteção que
   suspende as execuções. Isso faz sentido: a própria lei já suspende
   as execuções a partir do processamento — a tutela e a concessão são
   parte do mesmo momento, e não uma preparação que vem antes.


#### Resposta final (pergunta 8)

A tutela de urgência **antecedente ao pedido principal** é quase
inexistente: apenas 12 dos 5.709 processos de RJ (0,2%) tiveram tutela
deferida antes ou no dia do ajuizamento, e a via autônoma do art. 303 do
CPC (tutela cautelar em autos próprios) contribui com um único caso
vinculado a uma RJ — ainda por cima posterior ao pedido. O que existe na
prática é outra coisa: **13,9% das RJs registram tutela de urgência nos
próprios autos**. Em suma: o instrumento da tutela antecedente autônoma,
disponível desde 2015 no CPC, não se firmou como preparação da RJ no
corpus nacional; a suspensão das execuções que viabiliza a negociação é
obtida dentro da própria recuperação. Ressalva: a taxa mede
**deferimentos** (atos datados) — pedidos denegados são invisíveis ao
padrão Datajud.